In [ ]:
import numpy as np
import math
from math import sqrt
import cmath
import scipy
import pandas as pd
import csv
import os
from numpy import linalg
!pip install qutip
from qutip import *
from numpy.linalg import matrix_rank
from collections import Counter
import matplotlib.pyplot as plt
import scipy.stats as stats
import pandas as pd
import time

# For viewing vectors and array in matrix form
#import sympy
from sympy import Matrix

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
# defining functions for checking positive-semi-definiteness 
def PSD(M, tol):
    eigen_val = np.linalg.eigvalsh(M)
    if np.all(eigen_val >= -tol ) == True :
        return True 
    else :
        return False 

def check_valid_rho(M, tol_psd):
    shape = M.shape
    eigen_values = np.linalg.eigvalsh(M)
    psd = PSD(M, tol_psd)
    trace = M.trace()
    herm = np.all(M == M.conj().T)
    
    return shape, eigen_values, psd, trace, herm

def check_pure_rho(M, tol_idem):
    
    rank = np.linalg.matrix_rank(M)
    M_square = np.matmul(M,M)
    idempotent = np.all(abs(M_square - M) <= tol_idem )
    #idempotent = M_square == M
    trace = M_square.trace()
    
    return rank, idempotent, trace

def check_rho(M,tol_psd, tol_idem):
    
    '''
    print(" Shape: {0},\n PSD: {2} ,\n trace: {3} ,\n hermitian: {4} ,\n eigen values:\n {1} ,\n rank: {5},
          \n idempotent: {6} ,\n trace_rho_square: {7}".format(sh, eig, psd, tr, he, rank, idem, tr))
    '''
    sh, eig, psd, tr, he = check_valid_rho(M, tol_psd)
    rank, idem, tr =  check_pure_rho(M, tol_idem)
    return sh, eig, psd, tr, he, rank, idem, tr 


def fs_check(M):
    '''
    # Function to check the full separability of the generated state above 
    # based on the metrics in the paper https://arxiv.org/pdf/quant-ph/0005115.pdf
    '''
    # defining a list to store the ranks 
    rank_list=[]
    M = Qobj(M, dims=[[2,2,2],[2,2,2]])
    for i in range(0,3):
        # calculating the rank of the reduced density matrix of the ith subsystem
        rank_list.append(matrix_rank(M.ptrace([i]),hermitian=True))
    # if ranks of the all the 1 qubit subsystems are equal to 1 then it returns 1
    # meaning that the input Qobj represents a Fully-Separanle three qubit state
    if rank_list[0] == rank_list[1] == rank_list[2] == 1:
        return 1
    else:
        return 0 

def bs1_check(M):
    
    # defining a list to store the ranks 
    rank_list=[]
    M = Qobj(M, dims=[[2,2,2],[2,2,2]])
    for i in range(0,3):
        # calculating the rank of the reduced density matrix of the ith subsystem
        rank_list.append(matrix_rank(M.ptrace([i]),hermitian=True))
        
    # if ranks of the all the 1 qubit subsystems are equal to 1 then it returns 1
    # meaning that the input Qobj represents a Fully-Separanle three qubit state
    if rank_list[1] == rank_list[2] == 2 and rank_list[0] == 1:
        return 1
    else:
        return 0 
    
    
def bs2_check(M):
    rank_list=[]
    M = Qobj(M, dims=[[2,2,2],[2,2,2]])
    for i in range(0,3):
        rank_list.append(matrix_rank(M.ptrace([i]),hermitian=True))
    if rank_list[1] == 1 and rank_list[0] == rank_list[2] == 2:
        return 1
    else:
        return 0 
    
def bs3_check(M):

    rank_list=[]
    M = Qobj(M, dims=[[2,2,2],[2,2,2]])
    for i in range(0,3):
        rank_list.append(matrix_rank(M.ptrace([i]),hermitian=True))
    if rank_list[2] == 1 and rank_list[0] == rank_list[1] == 2:
        return 1
    else:
        return 0    

# Witness Based Classifiers

def wit_o(rho):
    rho = rho.reshape(8,8)
    O = 2*tensor(sigmax(),sigmax(),sigmax())
    O1 = 2*tensor(sigmax(),sigmax(),sigmaz())
    O2 = 2*tensor(sigmax(),sigmaz(),sigmax())
    O3 = 2*tensor(sigmaz(),sigmax(),sigmax())
    O_exp = np.matmul(rho, O.data.toarray()).trace()
    O1_exp = np.matmul(rho, O1.data.toarray()).trace()
    O2_exp = np.matmul(rho, O2.data.toarray()).trace()
    O3_exp = np.matmul(rho, O3.data.toarray()).trace()
    
    return [O_exp,O1_exp,O2_exp,O3_exp]

def wit_o_classifier(M):
    wit_o_values = wit_o(M)
    exp_O = wit_o_values[0]
    exp_O1 = wit_o_values[1]
    exp_O2 = wit_o_values[2]
    exp_O3 = wit_o_values[3]
    
    if exp_O != 0 :
        return 5 
    
    else : 
        if exp_O1 != 0 :
            if exp_02 != 0 : 
                return 4 
            else : 
                return 3
        else :
            if exp_O2 == 0 :
                if exp_O3 == 0 : 
                    return 0
                else :
                    return 1 
            else : 
                return 2 

def tangle_bipartition(rho):
    
    rho = Qobj(rho, dims = [[2,2,2],[2,2,2]])
    rho_reduced = {}
    bi_tangle = {}
    for i in range(3):
        key = 'rho'+str(i)
        rho_reduced[key] = rho.ptrace([i])
    rho_red_list = list(rho_reduced.values())
    bi_tangle['tangle(rho_0_12)'] = 4*np.linalg.det(rho_red_list[0])
    bi_tangle['tangle(rho_1_02)'] = 4*np.linalg.det(rho_red_list[1])
    bi_tangle['tangle(rho_2_01)'] = 4*np.linalg.det(rho_red_list[2])
                                      
    #i_tangle = dict((k, 4*np.linalg.det(v)) for k, v in rho_reduced.items())   

    return bi_tangle

def tangle_pairwise(rho):
    
    '''
     defined for three qubit pure states as given in the article 
     https://arxiv.org/pdf/quant-ph/9907047.pdf
    '''
    rho_red = []
    rho_red_tilda = []
    rho_red_product = []
    rho_red_pro_eig = []
    rho_red_pro_trace = []
    rho_red_tangle = []
    tangle_pair = []
    
    pair_tangle = {}
    j=[1,2,0]
    
    U = tensor(sigmay(),sigmay())
    Ud = U.dag()
    rho = Qobj(rho.data.toarray(), dims = [[2,2,2],[2,2,2]])
    for i in range(3):
        
        rho_red.append(rho.ptrace([i,j[i]])) 
        rho_red_tilda.append(U*(rho_red[i].conj())*Ud)
        rho_red_product.append(rho_red[i]*rho_red_tilda[i])
        evalue, evector = (rho_red_product[i]).eigenstates()
        rho_red_pro_eig.append(evalue)
        #rho_red_pro_trace.append(rho_red_product[i].tr())
        #rho_red_tangle.append(rho_red_pro_trace[i] -2*(rho_red_pro_eig[i][2]*rho_red_pro_eig[i][3]))
        rho_red_pro_eig[i] = sorted(list(rho_red_pro_eig[i]),reverse=True)
        k = rho_red_pro_eig[i]
        tangle_pair.append((max(k[0]-(k[1]+k[2]+k[3]),0))**2)
    
    pair_tangle['tangle(rho_01)'] = tangle_pair[0]
    pair_tangle['tangle(rho_12)'] = tangle_pair[1]
    pair_tangle['tangle(rho_20)'] = tangle_pair[2]
    
    return pair_tangle

def three_tangle(rho):
    three_t = []
    bi_t , pi_t = tangle_bipartition(rho) , tangle_pairwise(rho)
    bi_t_list, pi_t_list = list(bi_t.values()), list(pi_t.values())
    three_t = [bi_t_list[0]-(pi_t_list[0]+pi_t_list[2]), bi_t_list[1]-(pi_t_list[1]+pi_t_list[0]), bi_t_list[2]-(pi_t_list[2]+pi_t_list[1])]   
    three_tangle_min = np.min(three_t)
    return three_tangle_min

def det_rho_abc(M):
    det_list=[]
    for i in range(0,3):
        det_list.append(np.real(linalg.det(M.ptrace([i])))) 
        #det_list = np.real(det_list)              
    return det_list[0], det_list[1], det_list[2]

def three_tangle_ASD(psi):
    if psi.shape == (8,):
        t000 = psi[0]
        t111 = psi[7]
        tau_abc = 4*(t000**2)*(t111**2)
    
    elif psi.shape == (8,8):
        t000 = psi[0][0]
        t111 = psi[7][7]
        tau_abc = 4*(t000)*(t111)
    
    else : 
        None 
    return tau_abc 

def class_entropy_3T(M):
    three_t = three_tangle_ASD(M)
    M = Qobj(M, dims=[[2,2,2],[2,2,2]])
    '''
    replace with the following rank based classification
    s_a = matrix_rank(M.ptrace([0]), hermitian = True)
    s_b = matrix_rank(M.ptrace([1]), hermitian = True)
    s_c = matrix_rank(M.ptrace([2]), hermitian = True)
    '''
    s_a, s_b, s_c = det_rho_abc(M)
    return s_a, s_b, s_c, three_t
    #return list_all


# Reference Paper - https://arxiv.org/pdf/2203.09610.pdf


def FS_ASD():
    k = random.choice([0,1,2])
    #k= 0
    phi = (2*np.pi)*(-1)*np.random.default_rng().uniform(-1,0)
    psi = [0+0*1j]*8
    psi[4] = np.exp(1j*phi)*(-1)*np.random.default_rng().uniform(-1,0)
    
    if k == 0 :  
        psi[0] = np.random.random()
    elif k == 1 :
        psi[6] = np.random.random()
    elif k == 2 : 
        psi[5] = np.random.random()
    else :
        return None 
   
    #psi = np.array(map(lambda x: x/np.linalg.norm(psi), psi))    
    psi = np.array(psi/np.linalg.norm(psi))    
    rho = ket2dm(Qobj(psi, dims = [[2,2,2],[1,1,1]]))
    return rho.data.toarray()

def BS1_ASD():
    phi = (2*np.pi)*(-1)*np.random.default_rng().uniform(-1,0)
    psi = [0+0*1j]*8
    psi[4] = np.exp(1j*phi)*(-1)*np.random.default_rng().uniform(-1,0)
    psi[5] = (-1)*np.random.default_rng().uniform(-1,0)
    psi[6] = (-1)*np.random.default_rng().uniform(-1,0)
    psi[7] = (-1)*np.random.default_rng().uniform(-1,0)
    
    psi = np.array(psi/np.linalg.norm(psi))
    rho = ket2dm(Qobj(psi, dims = [[2,2,2],[1,1,1]]))
    return rho.data.toarray()
    
def BS2_ASD():
    phi = (2*np.pi)*(-1)*np.random.default_rng().uniform(-1,0)
    psi = [0+0*1j]*8
    
    psi[0] = (-1)*np.random.default_rng().uniform(-1,0)               # 000 
    psi[4] = np.exp(1j*phi)*(-1)*np.random.default_rng().uniform(-1,0)# 100  
    psi[5] = (-1)*np.random.default_rng().uniform(-1,0)               # 101
    psi = np.array(psi/np.linalg.norm(psi))
    rho = ket2dm(Qobj(psi, dims = [[2,2,2],[1,1,1]]))
    
    return rho.data.toarray()

def BS3_ASD():
    phi = (2*np.pi)*(-1)*np.random.default_rng().uniform(-1,0)
    psi = [0+0*1j]*8
    
    psi[0] = (-1)*np.random.default_rng().uniform(-1,0)               # 000 
    psi[4] = np.exp(1j*phi)*(-1)*np.random.default_rng().uniform(-1,0)# 100  
    psi[6] = (-1)*np.random.default_rng().uniform(-1,0)               # 110
    psi = np.array(psi/np.linalg.norm(psi))
    rho = ket2dm(Qobj(psi, dims = [[2,2,2],[1,1,1]]))
    
    return rho.data.toarray()

def W_ASD():
    phi = (2*np.pi)*(-1)*np.random.default_rng().uniform(-1,0)
    psi = [0+0*1j]*8
    
    psi[0] = (-1)*np.random.default_rng().uniform(-1,0)               # 000 
    psi[4] = np.exp(1j*phi)*(-1)*np.random.default_rng().uniform(-1,0)# 100  
    psi[5] = (-1)*np.random.default_rng().uniform(-1,0)               # 101
    psi[6] = (-1)*np.random.default_rng().uniform(-1,0)               # 110
    psi = np.array(psi/np.linalg.norm(psi))
    rho = ket2dm(Qobj(psi, dims = [[2,2,2],[1,1,1]]))
    
    return rho.data.toarray()

def GHZ_ASD():
    phi = (2*np.pi)*(-1)*np.random.default_rng().uniform(-1,0)
    psi = [0+0*1j]*8
    
    psi[0] = (-1)*np.random.default_rng().uniform(-1,0)               # 000 
    psi[4] = np.exp(1j*phi)*(-1)*np.random.default_rng().uniform(-1,0)# 100  
    psi[5] = (-1)*np.random.default_rng().uniform(-1,0)               # 101
    psi[6] = (-1)*np.random.default_rng().uniform(-1,0)               # 110
    psi[7] = (-1)*np.random.default_rng().uniform(-1,0)               # 111
    
    psi = np.array(psi/np.linalg.norm(psi))
    rho = ket2dm(Qobj(psi, dims = [[2,2,2],[1,1,1]]))
    
    return rho.data.toarray()

In [ ]:
import random

In [ ]:
index_map = {0:'000', 1: '001', 2:'010', 3:'011', 4:'100', 5:'101', 6:'110', 7:'111'} 
index_map_list = []
idm = {'000':0, '001':1, '010':2, '011':3, '100':4, '101':5, '110':6, '111':7}
idm['000'], int('000', 2), 0b000

(0, 0, 0)

In [ ]:
M = FS_ASD()
tol_psd = 1e-10
tol_idem = 1e-10
sh, eig, psd, tr, he, rank, idem, tr = check_rho(M ,tol_psd, tol_idem)
print(" Shape: {0},\n PSD: {2} ,\n trace: {3} ,\n hermitian: {4} ,\n eigen values:\n {1} ,\n rank: {5},\n idempotent: {6} ,\n trace_rho_square: {7}".format(sh, eig, psd, tr, he, rank, idem, tr))

print(" FS: {0},\n BS1: {1},\n BS2: {2},\n BS3: {3}".format(fs_check(M), bs1_check(M), bs2_check(M), bs3_check(M))) 

sa, sb, sc, three_t = class_entropy_3T(M)
print(" S_a: {0},\n S_b: {1},\n S_c: {2},\n three_t: {3}".format(sa, sb, sc, three_t))

 Shape: (8, 8),
 PSD: True ,
 trace: (1.0000000000000004+0j) ,
 hermitian: True ,
 eigen values:
 [-6.9388939e-18  0.0000000e+00  0.0000000e+00  0.0000000e+00
  0.0000000e+00  0.0000000e+00  0.0000000e+00  1.0000000e+00] ,
 rank: 1,
 idempotent: True ,
 trace_rho_square: (1.0000000000000004+0j)
 FS: 1,
 BS1: 0,
 BS2: 0,
 BS3: 0
 S_a: 0.0,
 S_b: 0.0,
 S_c: -4.49977272105161e-18,
 three_t: 0j


In [ ]:
M = BS1_ASD()
tol_psd = 1e-10
tol_idem = 1e-10
sh, eig, psd, tr, he, rank, idem, tr = check_rho(M ,tol_psd, tol_idem)
print(" Shape: {0},\n PSD: {2} ,\n trace: {3} ,\n hermitian: {4} ,\n eigen values:\n {1} ,\n rank: {5},\n idempotent: {6} ,\n trace_rho_square: {7}".format(sh, eig, psd, tr, he, rank, idem, tr))

print(" FS: {0},\n BS1: {1},\n BS2: {2},\n BS3: {3}".format(fs_check(M), bs1_check(M), bs2_check(M), bs3_check(M))) 

sa, sb, sc, three_t = class_entropy_3T(M)
print(" S_a: {0},\n S_b: {1},\n S_c: {2},\n three_t: {3}".format(sa, sb, sc, three_t))

 Shape: (8, 8),
 PSD: True ,
 trace: (1.0000000000000002+2.9773358122057117e-20j) ,
 hermitian: True ,
 eigen values:
 [-1.25162805e-17  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  1.93560342e-19  1.17824490e-16  1.00000000e+00] ,
 rank: 1,
 idempotent: True ,
 trace_rho_square: (1.0000000000000002+2.9773358122057117e-20j)
 FS: 0,
 BS1: 1,
 BS2: 0,
 BS3: 0
 S_a: 0.0,
 S_b: 0.0025147866486662333,
 S_c: 0.0025147866486662585,
 three_t: 0j


In [ ]:
M = BS2_ASD()
tol_psd = 1e-10
tol_idem = 1e-10
sh, eig, psd, tr, he, rank, idem, tr = check_rho(M ,tol_psd, tol_idem)
print(" Shape: {0},\n PSD: {2} ,\n trace: {3} ,\n hermitian: {4} ,\n eigen values:\n {1} ,\n rank: {5},\n idempotent: {6} ,\n trace_rho_square: {7}".format(sh, eig, psd, tr, he, rank, idem, tr))

print(" FS: {0},\n BS1: {1},\n BS2: {2},\n BS3: {3}".format(fs_check(M), bs1_check(M), bs2_check(M), bs3_check(M))) 

sa, sb, sc, three_t = class_entropy_3T(M)
print(" S_a: {0},\n S_b: {1},\n S_c: {2},\n three_t: {3}".format(sa, sb, sc, three_t))

 Shape: (8, 8),
 PSD: True ,
 trace: (0.9999999999999996+1.0274733126808812e-18j) ,
 hermitian: True ,
 eigen values:
 [-3.53389265e-17 -8.13681084e-34  0.00000000e+00  0.00000000e+00
  1.39605603e-34  4.74972578e-18  4.44669886e-17  1.00000000e+00] ,
 rank: 1,
 idempotent: True ,
 trace_rho_square: (0.9999999999999996+1.0274733126808812e-18j)
 FS: 0,
 BS1: 0,
 BS2: 1,
 BS3: 0
 S_a: 0.038515103550746436,
 S_b: 0.0,
 S_c: 0.0385151035507464,
 three_t: 0j


In [ ]:
M = BS3_ASD()
tol_psd = 1e-10
tol_idem = 1e-10
sh, eig, psd, tr, he, rank, idem, tr = check_rho(M ,tol_psd, tol_idem)
print(" Shape: {0},\n PSD: {2} ,\n trace: {3} ,\n hermitian: {4} ,\n eigen values:\n {1} ,\n rank: {5},\n idempotent: {6} ,\n trace_rho_square: {7}".format(sh, eig, psd, tr, he, rank, idem, tr))

print(" FS: {0},\n BS1: {1},\n BS2: {2},\n BS3: {3}".format(fs_check(M), bs1_check(M), bs2_check(M), bs3_check(M))) 

sa, sb, sc, three_t = class_entropy_3T(M)
print(" S_a: {0},\n S_b: {1},\n S_c: {2},\n three_t: {3}".format(sa, sb, sc, three_t))

 Shape: (8, 8),
 PSD: True ,
 trace: (0.9999999999999994+5.85007707809078e-19j) ,
 hermitian: True ,
 eigen values:
 [-2.75241887e-17  0.00000000e+00  0.00000000e+00  5.93432133e-35
  7.59177013e-34  2.58466766e-19  4.11435098e-17  1.00000000e+00] ,
 rank: 1,
 idempotent: True ,
 trace_rho_square: (0.9999999999999994+5.85007707809078e-19j)
 FS: 0,
 BS1: 0,
 BS2: 0,
 BS3: 1
 S_a: 0.1959730358980008,
 S_b: 0.19597303589800075,
 S_c: 0.0,
 three_t: 0j


In [ ]:
M = W_ASD()
tol_psd = 1e-10
tol_idem = 1e-10
sh, eig, psd, tr, he, rank, idem, tr = check_rho(M ,tol_psd, tol_idem)
print(" Shape: {0},\n PSD: {2} ,\n trace: {3} ,\n hermitian: {4} ,\n eigen values:\n {1} ,\n rank: {5},\n idempotent: {6} ,\n trace_rho_square: {7}".format(sh, eig, psd, tr, he, rank, idem, tr))

print(" FS: {0},\n BS1: {1},\n BS2: {2},\n BS3: {3}".format(fs_check(M), bs1_check(M), bs2_check(M), bs3_check(M))) 

sa, sb, sc, three_t = class_entropy_3T(M)

print(" S_a: {0},\n S_b: {1},\n S_c: {2},\n three_t: {3}".format(sa, sb, sc, three_t))

 Shape: (8, 8),
 PSD: True ,
 trace: (0.9999999999999996+3.0407386361178485e-20j) ,
 hermitian: True ,
 eigen values:
 [-1.11771227e-16  0.00000000e+00  5.87996717e-34  4.03423515e-33
  3.60655824e-18  2.01120618e-17  5.33581377e-17  1.00000000e+00] ,
 rank: 1,
 idempotent: True ,
 trace_rho_square: (0.9999999999999996+3.0407386361178485e-20j)
 FS: 0,
 BS1: 0,
 BS2: 0,
 BS3: 0
 S_a: 0.20111878145367262,
 S_b: 0.2306752149579937,
 S_c: 0.1484197035176196,
 three_t: 0j


In [ ]:
M = GHZ_ASD()
tol_psd = 1e-10
tol_idem = 1e-10
sh, eig, psd, tr, he, rank, idem, tr = check_rho(M ,tol_psd, tol_idem)
print(" Shape: {0},\n PSD: {2} ,\n trace: {3} ,\n hermitian: {4} ,\n eigen values:\n {1} ,\n rank: {5},\n idempotent: {6} ,\n trace_rho_square: {7}".format(sh, eig, psd, tr, he, rank, idem, tr))

print(" FS: {0},\n BS1: {1},\n BS2: {2},\n BS3: {3}".format(fs_check(M), bs1_check(M), bs2_check(M), bs3_check(M))) 

sa, sb, sc, three_t = class_entropy_3T(M)
print(" S_a: {0},\n S_b: {1},\n S_c: {2},\n three_t: {3}".format(sa, sb, sc, three_t))

 Shape: (8, 8),
 PSD: True ,
 trace: (1.0000000000000002+2.5040358161596974e-18j) ,
 hermitian: True ,
 eigen values:
 [-1.10783247e-16 -2.87859861e-17 -9.12878842e-33 -2.05953274e-34
  1.70123472e-18  1.74377788e-17  7.87968561e-17  1.00000000e+00] ,
 rank: 1,
 idempotent: True ,
 trace_rho_square: (1.0000000000000002+2.5040358161596974e-18j)
 FS: 0,
 BS1: 0,
 BS2: 0,
 BS3: 0
 S_a: 0.07660240299866561,
 S_b: 0.1772279177245174,
 S_c: 0.16183956355072798,
 three_t: (0.04867591950399107+0j)


In [ ]:
# SEP Mean values

instances = 10000
sep_list = [[] for i in range(4)]
for i in range(instances):
  M = FS_ASD()
  tol_psd = 1e-10
  tol_idem = 1e-10
  sa, sb, sc, three_t = class_entropy_3T(M)
  sep_list[0].append(sa)
  sep_list[1].append(sb)
  sep_list[2].append(sc)
  sep_list[3].append(three_t)

print(" mean S_a: {0},\n mean S_b: {1},\n mean S_c: {2},\n mean three_t: {3}".format(np.mean(sep_list[0]), np.mean(sep_list[1]), np.mean(sep_list[2]), np.mean(sep_list[3])))

 mean S_a: -3.051767686075226e-21,
 mean S_b: -1.5316728796465768e-17,
 mean S_c: 5.05097737997616e-20,
 mean three_t: 0j


In [ ]:
# BS1 Mean values

instances = 10000
bs1_list = [[] for i in range(4)]
for i in range(instances):
  M = BS1_ASD()
  sa, sb, sc, three_t = class_entropy_3T(M)
  bs1_list[0].append(sa)
  bs1_list[1].append(sb)
  bs1_list[2].append(sc)
  bs1_list[3].append(three_t)

print(" mean S_a: {0},\n mean S_b: {1},\n mean S_c: {2},\n mean three_t: {3}".format(np.mean(bs1_list[0]), np.mean(bs1_list[1]), np.mean(bs1_list[2]), np.mean(bs1_list[3])))

 mean S_a: 0.0,
 mean S_b: 0.09625423325425997,
 mean S_c: 0.09625423325425995,
 mean three_t: 0j


In [ ]:
# BS2 Mean values

instances = 10000
bs2_list = [[] for i in range(4)]
for i in range(instances):
  M = BS2_ASD()
  sa, sb, sc, three_t = class_entropy_3T(M)
  bs2_list[0].append(sa)
  bs2_list[1].append(sb)
  bs2_list[2].append(sc)
  bs2_list[3].append(three_t)

print(" mean S_a: {0},\n mean S_b: {1},\n mean S_c: {2},\n mean three_t: {3}".format(np.mean(bs2_list[0]), np.mean(bs2_list[1]), np.mean(bs2_list[2]), np.mean(bs2_list[3])))

 mean S_a: 0.0754792860659922,
 mean S_b: 0.0,
 mean S_c: 0.0754792860659922,
 mean three_t: 0j


In [ ]:
# BS3 Mean values

instances = 10000
bs3_list = [[] for i in range(4)]
for i in range(instances):
  M = BS3_ASD()
  sa, sb, sc, three_t = class_entropy_3T(M)
  bs3_list[0].append(sa)
  bs3_list[1].append(sb)
  bs3_list[2].append(sc)
  bs3_list[3].append(three_t)

print(" mean S_a: {0},\n mean S_b: {1},\n mean S_c: {2},\n mean three_t: {3}".format(np.mean(bs3_list[0]), np.mean(bs3_list[1]), np.mean(bs3_list[2]), np.mean(bs3_list[3])))

 mean S_a: 0.07671448849596772,
 mean S_b: 0.07671448849596772,
 mean S_c: 0.0,
 mean three_t: 0j


In [ ]:
# W Mean values

instances = 10000
w_list = [[] for i in range(4)]
for i in range(instances):
  M = W_ASD()
  sa, sb, sc, three_t = class_entropy_3T(M)
  w_list[0].append(sa)
  w_list[1].append(sb)
  w_list[2].append(sc)
  w_list[3].append(three_t)

print(" mean S_a: {0},\n mean S_b: {1},\n mean S_c: {2},\n mean three_t: {3}".format(np.mean(w_list[0]), np.mean(w_list[1]), np.mean(w_list[2]), np.mean(w_list[3])))

 mean S_a: 0.09586545797180857,
 mean S_b: 0.09531953452959001,
 mean S_c: 0.09522021055260525,
 mean three_t: 0j


In [ ]:
# GHZ Mean values

instances = 10000
ghz_list = [[] for i in range(4)]
for i in range(instances):
  M = GHZ_ASD()
  sa, sb, sc, three_t = class_entropy_3T(M)
  ghz_list[0].append(sa)
  ghz_list[1].append(sb)
  ghz_list[2].append(sc)
  ghz_list[3].append(three_t)

print(" mean S_a: {0},\n mean S_b: {1},\n mean S_c: {2},\n mean three_t: {3}".format(np.mean(ghz_list[0]), np.mean(ghz_list[1]), np.mean(ghz_list[2]), np.mean(ghz_list[3])))

 mean S_a: 0.0975982748693879,
 mean S_b: 0.13010006284971942,
 mean S_c: 0.12926283775606975,
 mean three_t: (0.12761764566218442+0j)


In [2]:
import math
math.log(2)

0.6931471805599453

In [ ]:
print(" std S_a: {0},\n std S_b: {1},\n std S_c: {2},\n std three_t: {3}".format(np.std(ghz_list[0]), np.std(ghz_list[1]), np.std(ghz_list[2]), np.std(ghz_list[3])))

 std S_a: 0.07348819705446649,
 std S_b: 0.06665214681551372,
 std S_c: 0.06639599816322993,
 std three_t: 0.16155823700193309


In [ ]:
#https://stackoverflow.com/questions/6360286/why-does-foo-appendbar-affect-all-elements-in-a-list-of-lists
'''
test = []
dum = []
for i in range(4):
  test.append(dum[:])
test[0].append(a)
test

dum = []
test2 = [dum[:] for i in range(4)]
test2
test2[0].append(a)
test2

test3 = [[] for i in range(4)]
test3
test3[0].append(a)
test3
'''

'\ntest = []\ndum = []\nfor i in range(4):\n  test.append(dum[:])\ntest[0].append(a)\ntest\n\ndum = []\ntest2 = [dum[:] for i in range(4)]\ntest2\ntest2[0].append(a)\ntest2\n\ntest3 = [[] for i in range(4)]\ntest3\ntest3[0].append(a)\ntest3\n'